# Lesson 17 · Transformer & GPT: build a small GPT that continues *The Tale of Kiều*, then look inside the real GPT-2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/17-transformer-en.ipynb)

This notebook accompanies [Lesson 17 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/17-transformer.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
We write a GPT ourselves in numpy, built exactly like GPT-2: token and position embeddings, multi-head masked
self-attention, pre-LN blocks with skip connections and an MLP, a softmax at the output, and a handwritten backward
pass. We train it right in the notebook on *The Tale of Kiều* (*Truyện Kiều*, Nguyễn Du's Vietnamese poem), one
character at a time, then set it beside the measurements on the real GPT-2 small (124 million parameters) that the
demo uses.

**What you'll do:**
1. Turn characters into vectors: token embedding plus position embedding.
2. Write masked self-attention (as a loop first, then with matrices), multiple heads and the whole Transformer block,
   and check the backward pass with finite differences.
3. Train the small GPT with AdamW, check that the mask really hides the future, then generate verse with temperature
   and top-k.
4. Write a KV cache, check that it gives exactly the result of the full pass, then count GPT-2's compute and memory.
5. Grow an induction head in a 2-block model, then read the real GPT-2's heads, its logit lens and its distribution
   over 50,257 tokens.

**Background:** [Lesson 14 · Tokens & embeddings](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-embedding/),
[Lesson 16 · The attention mechanism](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/attention/),
[Lesson 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chuan-hoa/) and
[Lesson 13 · Skip connections & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/).
**Run time:** under a minute on a CPU, including the first data download (~0.4 MB).

Every number about the small GPT is computed by the cells below. The measurements on GPT-2 are read from the very
results files the demo uses (the notebook doesn't download GPT-2's 548 MB of weights); GPT-2's parameter count,
compute and memory, though, we recompute ourselves from its published configuration, and they match the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import hashlib
import json
import math
import re
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
SEQ = LinearSegmentedColormap.from_list("white_blue", ["#f7f7f7", BLUE, "#0b2a59"])  # 0 → large (attention weights)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it
    is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def sci(x):
    """A very small number: 2.4·10⁻⁷ (0 prints as 0; from 0.01 up, a plain decimal, like 0.12)."""
    if x == 0:
        return "0"
    if abs(x) >= 0.01:
        return num(x)
    e = math.floor(math.log10(abs(x)))
    return f"{num(x / 10**e, 1)}·10{str(e).translate(SUP)}"


def show(M, digits=3):
    """Print a small matrix in the English number format."""
    print("\n".join("  ".join(f"{num(v, digits):>6}" for v in row) for row in M))


print(num(124439808, 0), num(0.1, 1), num(-4.3805, 4), pct(0.446), sci(2.4e-7))

## 2 · The data: *The Tale of Kiều*, one character at a time

We use *The Tale of Kiều* (*Truyện Kiều*) by Nguyễn Du (early 19th century), a Vietnamese poem in the public
domain. The copy comes from the [Vietnamese Wikisource](https://vi.wikisource.org/w/index.php?oldid=205132), pinned
at revision 205132 and checked by its MD5 hash, exactly as in Lesson 15, so everyone who runs it gets exactly the
same text. Wikimedia rejects requests without a User-Agent, so the cell below sets one before downloading.

The cell also loads two results files that the demo uses (`fetch_json`, with an MD5 check): the measurements on the
real GPT-2 small, computed once in numpy on the model's weights. We read them in section 12.

In [ ]:
SRC = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"  # The Tale of Kiều, revision 205132
RAW_MD5 = "d4090daac094841888ee2b2b18661065"
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (education; https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # every download below sends this User-Agent
RAW = Path("truyen_kieu_205132.wiki")
if not RAW.exists():
    urllib.request.urlretrieve(SRC, RAW)
assert hashlib.md5(RAW.read_bytes()).hexdigest() == RAW_MD5, f"{RAW} is corrupted; delete it and download it again"


# Measurements on the real GPT-2 (the same files as the demo on the web).
# When publishing, build.py replaces the COURSE_DATA_URL placeholder below with the data/ folder of the public
# notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


inside = fetch_json("gpt2-inside.json", "9ee32934381ae700b3d18a63059a20b6")  # predictions, logit lens, heads
samp = fetch_json("gpt2-sampling.json", "20257a3d7e84893538a9851f5712b268")  # next-token distributions
size = sum(f.stat().st_size for f in (RAW, Path("data/gpt2-inside.json"), Path("data/gpt2-sampling.json")))
print(f"3 files ready, {num(size / 1e6, 1)} MB in total")

The downloaded file is the page's wiki source (wikitext). We take the part between the two `<poem>` tags and drop
the line numbers `{{số|…}}` (*số*, “number”) and the empty lines, as in Lesson 15. The vocabulary is every character
that occurs in the text; each character is one token, and its token ID is its position in the sorted list. As in
Lesson 15, the first 3,000 lines of verse are for training and the last 254 lines for validation.

In [ ]:
body = RAW.read_bytes().decode("utf-8").split("<poem>")[1].split("</poem>")[0]
lines = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]  # drop the line numbers
text = "\n".join(ln for ln in lines if ln) + "\n"
assert hashlib.md5(text.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8"
LINES = text[:-1].split("\n")
CHARS = sorted(set(text))  # the vocabulary: every distinct character, space and "\n" included
V = len(CHARS)
stoi = {ch: i for i, ch in enumerate(CHARS)}
ids = np.array([stoi[ch] for ch in text])
cut = len("\n".join(LINES[:3000])) + 1
train_ids, val_ids = ids[:cut], ids[cut:]  # first 3,000 lines / last 254 lines


def decode(seq):
    return "".join(CHARS[i] for i in seq)


print(f"{num(len(LINES), 0)} lines of verse, {num(len(text), 0)} characters, a vocabulary of {V} characters")
print(
    f"training: first 3,000 lines, {num(len(train_ids), 0)} characters · validation: last {len(LINES) - 3000} lines, {num(len(val_ids), 0)} characters"
)
print("token IDs of 'T', 'r', 'ă', 'm':", [stoi[c] for c in "Trăm"])
print("\n".join(LINES[:4]))
assert (len(LINES), len(text), V, len(train_ids), len(val_ids)) == (3254, 104204, 132, 96204, 8000)

A language model learns just one thing: read the characters so far and predict the next one. Before building the
Transformer, we set two baselines to compare against. Guessing uniformly over 132 characters costs $\log_2 132$ bits
per character. A **bigram** model looks only at the immediately preceding character: count the pairs in the
training set and add a little $\alpha$ so that no pair has probability 0. The measure is **bits per character**
(base-2 cross-entropy, Lesson 04) on the validation set: the lower the better.

In [ ]:
uniform_bits = math.log2(V)
counts = np.zeros((V, V))
np.add.at(counts, (train_ids[:-1], train_ids[1:]), 1)  # counts[a, b] = how many times b comes right after a
for alpha in (0.01, 0.1, 1.0):
    P_bi = (counts + alpha) / (counts + alpha).sum(1, keepdims=True)
    print(f"bigram, α = {num(alpha, 2)}: {num(-np.log2(P_bi[val_ids[:-1], val_ids[1:]]).mean(), 3)} bits/character")
P_bigram = (counts + 0.01) / (counts + 0.01).sum(1, keepdims=True)
bigram_bits = -np.log2(P_bigram[val_ids[:-1], val_ids[1:]]).mean()
print(f"uniform guess: log₂ {V} = {num(uniform_bits, 3)} bits/character")

The bigram is best at $\alpha = 0.01$: 3.14 bits, much better than the 7.04 bits of the uniform guess. But it
remembers nothing beyond one character, so it doesn't know whether it is in the middle of a word or at the end of a
six-syllable or an eight-syllable line. To use the whole passage before it, each position has to *take information
from the positions before it*. That is the job of self-attention.

## 3 · The input: token embedding plus position embedding

As in Lesson 14, each token looks up one row of the embedding table $E$ (of size $V \times d$). GPT also has a
second table $P$, one learned vector per **position**, and adds the two vectors:

$$h_0[t] = E[x_t] + P[t]$$

Our small GPT has $d = 48$ and a context window of 64 characters; GPT-2 small has $d = 768$, a window of 1,024
tokens and 50,257 tokens in its vocabulary. Self-attention *without a mask* knows nothing at all about order:
permute the input tokens and the output is just permuted the same way (in Lesson 16, the reader without positions
answers 23 for every order of “hai mươi ba”, the Vietnamese words for twenty-three). With $P$, the same letter “a”
at position 3 and at position 40 gives two different vectors. (The causal mask of section 4 by itself also leaks a
little position information: Haviv et al., 2022, trained GPTs without $P$. GPT-2 still uses $P$.)

Initialization: $E \sim \mathcal N(0, 1)$, $P \sim \mathcal N(0, 0.1^2)$, each matrix $\mathcal N(0, 1/n_\text{in})$,
as in the model of Lesson 23. The two matrices that write into the residual stream ($W_O$, $W_2$) are also multiplied
by $1/\sqrt{2L}$, following the GPT-2 paper (multiply by $1/\sqrt N$ for $N$ residual layers), so that the sum of
many blocks doesn't blow up. The output matrix starts at 0.

In [ ]:
CFG = {"V": V, "ctx": 64, "d": 48, "L": 2, "heads": 4}  # vocabulary, context window, width, blocks, heads


def init_gpt(cfg, seed=0, dtype=np.float32):
    rng = np.random.default_rng(seed)
    d, L = cfg["d"], cfg["L"]
    r = 1 / math.sqrt(2 * L)

    def mat(n_in, n_out, scale=1.0):
        return rng.normal(0, scale / math.sqrt(n_in), (n_in, n_out)).astype(dtype)

    def vec(n, value=0.0):
        return np.full(n, value, dtype)

    p = {"wte": rng.normal(0, 1, (cfg["V"], d)).astype(dtype), "wpe": rng.normal(0, 0.1, (cfg["ctx"], d)).astype(dtype)}
    for l in range(L):
        p |= {f"{l}.g1": vec(d, 1), f"{l}.b1": vec(d)}  # LayerNorm before the attention layer
        p |= {f"{l}.Wqkv": mat(d, 3 * d), f"{l}.bqkv": vec(3 * d)}  # Q, K, V of every head, merged into one matrix
        p |= {f"{l}.Wo": mat(d, d, r), f"{l}.bo": vec(d)}  # W_O: concatenated heads → residual stream
        p |= {f"{l}.g2": vec(d, 1), f"{l}.b2": vec(d)}  # LayerNorm before the MLP
        p |= {f"{l}.W1": mat(d, 4 * d), f"{l}.c1": vec(4 * d)}  # MLP d → 4d
        p |= {f"{l}.W2": mat(4 * d, d, r), f"{l}.c2": vec(d)}  # 4d → d
    p |= {"gf": vec(d, 1), "bf": vec(d)}  # final LayerNorm
    p |= {"Wh": np.zeros((d, cfg["V"]), dtype), "bh": vec(cfg["V"])}  # output: d → V logits
    return p


p0 = init_gpt(CFG)
window = train_ids[: CFG["ctx"]]
h0 = p0["wte"][window] + p0["wpe"][: len(window)]  # look up both tables, then add
print("one window:", repr(decode(window[:36])), "…")
print("h₀:", h0.shape, "· E:", p0["wte"].shape, "· P:", p0["wpe"].shape)

## 4 · Masked self-attention: one head

As in Lesson 16, each position $t$ makes three row vectors from $h[t]$: a query $q_t = h[t]W_Q$ (“what do I
need?”), a key $k_t = h[t]W_K$ (“what do I have?”) and a value $v_t = h[t]W_V$ (“what do I pass on if I'm
chosen?”). Position $t$ scores every position $j$, divides by $\sqrt{d_h}$ ($d_h$ is the size of one head), applies
softmax, then takes the weighted average of the $v_j$:

$$s_{tj} = \frac{q_t \cdot k_j}{\sqrt{d_h}}, \qquad \alpha_{tj} = \operatorname{softmax}_j(s_{tj}), \qquad
o_t = \sum_j \alpha_{tj}\, v_j$$

What differs from Lesson 16: this is **self**-attention (the queries and the keys come from the same sequence), and
it has a **causal mask**: position $t$ may only look at $j \le t$. The cell below writes the computation twice: a
loop that follows the definition exactly, then the matrix form $\operatorname{softmax}(QK^\top/\sqrt{d_h} + M)\,V$,
where $M$ assigns $-\infty$ to every entry with $j > t$ (because $e^{-\infty} = 0$). We take the first head of
block 0 and skip LayerNorm to keep it short.

In [ ]:
def softmax(s):
    e = np.exp(s - s.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


def attention_loop(q, k, v):
    """One head, one position at a time: position t scores j = 0…t, softmax, weighted sum of the v_j."""
    n, dh = q.shape
    out = np.zeros_like(v)
    for t in range(n):
        s = np.array([q[t] @ k[j] for j in range(t + 1)]) / math.sqrt(dh)
        alpha = softmax(s)
        out[t] = sum(alpha[j] * v[j] for j in range(t + 1))
    return out


def attention(q, k, v, causal=True):
    """The same computation in matrix form; q, k, v may have extra leading axes for the heads (…, n, d_h)."""
    n, dh = q.shape[-2:]
    s = q @ np.swapaxes(k, -1, -2) / math.sqrt(dh)
    if causal:
        s = np.where(np.triu(np.ones((n, n), bool), 1), -np.inf, s)  # entries j > t: −∞
    A = softmax(s)
    return A @ v, A


d, nh = CFG["d"], CFG["heads"]
dh = d // nh
qkv = h0 @ p0["0.Wqkv"]  # (64, 3·48): Q, K, V of all 4 heads, side by side
q1, k1, v1 = qkv[:, :dh], qkv[:, d : d + dh], qkv[:, 2 * d : 2 * d + dh]  # the first head: 12 columns of each
o_loop = attention_loop(q1, k1, v1)
o_mat, A = attention(q1, k1, v1)
print(f"q, k, v each {q1.shape}; the loop and the matrix form differ by at most {sci(np.abs(o_loop - o_mat).max())}")
print("first 5 rows of the attention matrix A (row = query position t, column = position j):")
show(A[:5, :5])
assert np.allclose(o_loop, o_mat, atol=1e-5) and np.all(np.triu(A, 1) == 0)

Above the diagonal of $A$, every entry is *exactly* 0, and each row sums to 1. Row 0 has only one entry: the first
position can only see itself.

A special case helps show what the mask does: if all the scores are equal (for example $q = 0$), softmax splits
evenly, and $o_t$ is the **average** of $v_0, \dots, v_t$. The matrix $A$ is then the “running average” matrix that
Karpathy uses to open his lecture *Let's build GPT*:

In [ ]:
_, A_uniform = attention(np.zeros_like(q1[:4]), k1[:4], v1[:4])
show(A_uniform)

## 5 · Multiple heads, then the whole Transformer block

**Multiple heads.** Instead of one attention of size $d$, we split $d = 48$ into 4 heads of $d_h = 12$ dimensions
each; each head has its own $W_Q$, $W_K$, $W_V$ and chooses for itself where to look. In the code, the $Q$, $K$, $V$
of every head sit side by side in one matrix $W_{QKV}$ of size $d \times 3d$ (as in GPT-2), so one matrix
multiplication and a reshape are enough. The outputs of the heads are **concatenated** into a $d$-dimensional
vector, then multiplied by $W_O$. GPT-2 small does exactly this with 12 heads × 64 dimensions = 768.

The cell below splits $h_0 W_{QKV}$ into 4 heads and checks that the first head is exactly the head we just computed
by hand in section 4.

In [ ]:
n = len(window)
q, k, v = qkv.reshape(n, 3, nh, dh).transpose(1, 2, 0, 3)  # each (head, position, d_h) = (4, 64, 12)
o_heads, A_heads = attention(q, k, v)  # all 4 heads in one call
o_cat = o_heads.transpose(1, 0, 2).reshape(n, d)  # concatenate the 4 heads: (64, 48)
out = o_cat @ p0["0.Wo"]  # W_O mixes the heads, back to d dimensions
print("q:", q.shape, "→ heads concatenated:", o_cat.shape, "→ after W_O:", out.shape)
assert np.allclose(o_heads[0], o_mat)  # head 0 matches the single-head computation above

**The Transformer block.** GPT-2 stacks $L$ identical blocks. Each block has two sublayers; each sublayer is *added
to* the token's vector through a skip connection (Lesson 13), and LayerNorm (Lesson 10) sits at the **input** of the
sublayer (pre-LN; the original Transformer put it after the addition):

$$h \leftarrow h + \operatorname{MSA}(\operatorname{LN}_1(h)), \qquad
h \leftarrow h + \operatorname{GELU}(\operatorname{LN}_2(h)\,W_1 + c_1)\,W_2 + c_2$$

The attention layer is where the positions *exchange* information; the MLP ($d \to 4d \to d$, with GPT-2's $\tanh$
form of the GELU activation) processes **each position separately**. The vector $h$ that runs through the blocks is
called the **residual stream**: each sublayer reads it and adds a contribution. At the end come one more LayerNorm
and the output layer $z = hW_h + b_h$, which gives 132 logits. (GPT-2 has no separate $W_h$ but reuses $E^\top$; we
keep a separate $W_h$, as Llama does, so that the output layer can learn more freely.)

The cell below writes the whole forward pass. It takes a batch $X$ of size (B, n) and returns logits of size
(B, n, V): **one prediction at every position**. It stores every intermediate value, because the backward pass will
need them.

In [ ]:
C_GELU = math.sqrt(2 / math.pi)


def layernorm(x, g, b, eps=1e-5):
    xc = x - x.mean(-1, keepdims=True)
    inv = 1 / np.sqrt((xc * xc).mean(-1, keepdims=True) + eps)
    return xc * inv * g + b, (xc * inv, inv)


def gelu(u):
    """GELU in its tanh form, as in GPT-2."""
    t = np.tanh(C_GELU * (u + 0.044715 * u * u * u))
    return 0.5 * u * (1 + t), t


def forward(p, X, cfg, causal=True):
    B, n = X.shape
    d, nh = cfg["d"], cfg["heads"]
    h = p["wte"][X] + p["wpe"][:n]  # section 3
    layers = []
    for l in range(cfg["L"]):
        a, ln1 = layernorm(h, p[f"{l}.g1"], p[f"{l}.b1"])
        qkv = a @ p[f"{l}.Wqkv"] + p[f"{l}.bqkv"]
        q, k, v = qkv.reshape(B, n, 3, nh, d // nh).transpose(2, 0, 3, 1, 4)  # each (B, head, n, d_h)
        o, A = attention(q, k, v, causal)  # section 4, all heads at once
        o = o.transpose(0, 2, 1, 3).reshape(B, n, d)  # concatenate the heads
        h = h + o @ p[f"{l}.Wo"] + p[f"{l}.bo"]  # skip connection around the attention layer
        m, ln2 = layernorm(h, p[f"{l}.g2"], p[f"{l}.b2"])
        u = m @ p[f"{l}.W1"] + p[f"{l}.c1"]
        gu, t = gelu(u)
        h = h + gu @ p[f"{l}.W2"] + p[f"{l}.c2"]  # skip connection around the MLP
        layers.append({"a": a, "ln1": ln1, "q": q, "k": k, "v": v, "A": A, "o": o, "m": m, "ln2": ln2, "u": u})
        layers[-1] |= {"gu": gu, "t": t}
    hf, lnf = layernorm(h, p["gf"], p["bf"])
    return hf @ p["Wh"] + p["bh"], {"X": X, "layers": layers, "hf": hf, "lnf": lnf}


def cross_entropy(z, Y, w=None):
    """The mean loss (in nats) over the positions, optionally weighted by w, and the gradient ∂L/∂z."""
    z2, y = z.reshape(-1, z.shape[-1]), Y.ravel()
    w = np.ones(len(y), z.dtype) if w is None else w.ravel().astype(z.dtype)
    P = softmax(z2)
    loss = -(w * np.log(P[np.arange(len(y)), y])).sum() / w.sum()
    P[np.arange(len(y)), y] -= 1
    return float(loss), (P * (w / w.sum())[:, None]).reshape(z.shape)

Counting the small GPT's parameters: each block has $W_{QKV}$ ($3d^2$), $W_O$ ($d^2$), $W_1$ and $W_2$ ($4d^2$
each), that is $12d^2$, plus $13d$ biases and LayerNorm coefficients. The cell below counts them both ways and checks
that they agree. It also runs the forward pass for the first time.

### 🤔 Predict first
Before any training, what is the loss on a batch, in bits per character? (Hint: the output matrix $W_h$ starts at 0.)

In [ ]:
L_, V_ = CFG["L"], CFG["V"]
formula = L_ * (12 * d * d + 13 * d) + 2 * d + V_ * d + CFG["ctx"] * d + d * V_ + V_
n_params = sum(a.size for a in p0.values())
print(f"{len(p0)} parameter arrays, {num(n_params, 0)} numbers in total; the formula gives {num(formula, 0)}")
X0 = np.stack([train_ids[i : i + 64] for i in (0, 1000, 2000, 3000)])  # 4 windows of 64 characters each
Y0 = np.stack([train_ids[i + 1 : i + 65] for i in (0, 1000, 2000, 3000)])  # targets: each position's next character
z0, _ = forward(p0, X0, CFG)
loss0, _ = cross_entropy(z0, Y0)
print(
    f"logits: {z0.shape} · initial loss {num(loss0 / math.log(2), 4)} bits/character; log₂ {V} = {num(uniform_bits, 4)}"
)
assert n_params == formula == 72516 and abs(loss0 - math.log(V)) < 1e-5

Exactly $\log_2 132$: with $W_h = 0$ and $b_h = 0$, every logit is 0 and softmax splits evenly, so the freshly
initialized model is a “uniform guess”, however random those 72,516 parameters are. Initializing the output layer at
0 is a deliberate choice: the initial loss is exactly the uniform-guess baseline, no higher.

## 6 · A handwritten backward pass, checked with finite differences

The backward pass goes through exactly the steps of the forward pass, in reverse order (Lesson 06). There are three
new things compared with earlier lessons:
- **The attention softmax:** with $\alpha = \operatorname{softmax}(s)$ row by row,
  $\partial L/\partial s = \alpha \odot (\partial L/\partial \alpha - \sum_j \alpha_j\, \partial L/\partial \alpha_j)$.
  Masked entries have $\alpha = 0$, so they automatically get gradient 0: the mask needs no special handling.
- **Skip connections:** $h + F(h)$ has derivative $1 + F'$, so the gradient of the residual stream passes straight
  through, and the part through the sublayer is *added* to it (Lesson 13).
- **Embeddings:** the gradient of $E$ accumulates into the rows of the tokens used (`np.add.at`, because a character
  occurs many times).

In [ ]:
def layernorm_back(dy, g, cache):
    xh, inv = cache
    dxh = dy * g
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, (dy * xh).reshape(-1, xh.shape[-1]).sum(0), dy.reshape(-1, xh.shape[-1]).sum(0)


def gelu_back(dy, u, t):
    return dy * (0.5 * (1 + t) + 0.5 * u * (1 - t * t) * C_GELU * (1 + 3 * 0.044715 * u * u))


def flat(a):
    return a.reshape(-1, a.shape[-1])

In [ ]:
def backward(p, cache, dz, cfg):
    X, hf = cache["X"], cache["hf"]
    B, n = X.shape
    d, nh = cfg["d"], cfg["heads"]
    dh = d // nh
    g = {"Wh": flat(hf).T @ flat(dz), "bh": flat(dz).sum(0)}
    dr, g["gf"], g["bf"] = layernorm_back(dz @ p["Wh"].T, p["gf"], cache["lnf"])  # dr = ∂L/∂(residual stream)
    for l in reversed(range(cfg["L"])):
        c = cache["layers"][l]
        g[f"{l}.W2"], g[f"{l}.c2"] = flat(c["gu"]).T @ flat(dr), flat(dr).sum(0)  # MLP
        du = gelu_back(dr @ p[f"{l}.W2"].T, c["u"], c["t"])
        g[f"{l}.W1"], g[f"{l}.c1"] = flat(c["m"]).T @ flat(du), flat(du).sum(0)
        dx, g[f"{l}.g2"], g[f"{l}.b2"] = layernorm_back(du @ p[f"{l}.W1"].T, p[f"{l}.g2"], c["ln2"])
        dr = dr + dx  # skip connection: straight through + through the MLP
        g[f"{l}.Wo"], g[f"{l}.bo"] = flat(c["o"]).T @ flat(dr), flat(dr).sum(0)  # attention
        do = (dr @ p[f"{l}.Wo"].T).reshape(B, n, nh, dh).transpose(0, 2, 1, 3)
        A, q, k, v = c["A"], c["q"], c["k"], c["v"]
        dA, dv = do @ v.transpose(0, 1, 3, 2), A.transpose(0, 1, 3, 2) @ do
        ds = A * (dA - (dA * A).sum(-1, keepdims=True)) / math.sqrt(dh)  # through the softmax; masked entries: A = 0
        dq, dk = ds @ k, ds.transpose(0, 1, 3, 2) @ q
        dqkv = np.stack([dq, dk, dv]).transpose(1, 3, 0, 2, 4).reshape(B, n, 3 * d)
        g[f"{l}.Wqkv"], g[f"{l}.bqkv"] = flat(c["a"]).T @ flat(dqkv), flat(dqkv).sum(0)
        dx, g[f"{l}.g1"], g[f"{l}.b1"] = layernorm_back(dqkv @ p[f"{l}.Wqkv"].T, p[f"{l}.g1"], c["ln1"])
        dr = dr + dx
    g["wpe"] = np.zeros_like(p["wpe"])
    g["wpe"][:n] = dr.sum(0)
    g["wte"] = np.zeros_like(p["wte"])
    np.add.at(g["wte"], X.ravel(), flat(dr))
    return g

We check every parameter array with central differences $\frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$,
as in Lesson 06: a tiny GPT (vocabulary 7, window 5, $d = 8$, 2 blocks, 2 heads) in float64, with noise added to
the parameters so that no array sits at a special value (LayerNorm's $g = 1$, $W_h = 0$), and a loss with random
weights $w$.

The cell also prints the gradient of the key bias $b_K$ (part of $b_{QKV}$). Guess first: how large is it?

In [ ]:
cfg_t = {"V": 7, "ctx": 5, "d": 8, "L": 2, "heads": 2}
rng = np.random.default_rng(0)
pt = {k_: a + 0.3 * rng.standard_normal(a.shape) for k_, a in init_gpt(cfg_t, 1, np.float64).items()}
Xt, Yt, wt = rng.integers(0, 7, (3, 5)), rng.integers(0, 7, (3, 5)), rng.random((3, 5))
zt, ct = forward(pt, Xt, cfg_t)
gt = backward(pt, ct, cross_entropy(zt, Yt, wt)[1], cfg_t)
worst, eps = 0.0, 1e-5
for name, a in pt.items():
    for _ in range(4):  # 4 random entries of each array
        i = tuple(rng.integers(0, s) for s in a.shape)
        old = a[i]
        a[i] = old + eps
        lp = cross_entropy(forward(pt, Xt, cfg_t)[0], Yt, wt)[0]
        a[i] = old - eps
        lm = cross_entropy(forward(pt, Xt, cfg_t)[0], Yt, wt)[0]
        a[i] = old
        num_ = (lp - lm) / (2 * eps)
        worst = max(worst, abs(num_ - gt[name][i]) / max(1e-4, abs(num_) + abs(gt[name][i])))
print(f"{len(pt)} parameter arrays × 4 entries: largest relative error {sci(worst)}")
bk = np.abs(gt["0.bqkv"][cfg_t["d"] : 2 * cfg_t["d"]]).max()  # the key (K) part of the bias
print(f"largest gradient of the key bias b_K: {sci(bk)}")
assert worst < 1e-6 and bk < 1e-12, "the backward pass is wrong: check it again"

The backward pass matches the finite differences in every array, with a relative error below $10^{-6}$. The gradient
of $b_K$ is 0 up to rounding, and that is not a bug. Adding the same vector $b_K$ to every key raises every score in
a row by the same amount $q_t \cdot b_K$, and softmax doesn't change when all its inputs rise by the same amount. So
$b_K$ has no effect at all on the output (many later models drop the biases of Q, K, V altogether). Section 13 shows
that it still “drifts” when trained with Adam.

## 7 · Training the small GPT on *The Tale of Kiều*

Each step draws 16 random windows of 65 consecutive characters from the training set. The input $x$ is the first 64
characters; the target $y$ is the 64 characters that follow, shifted by one position. Because of the mask, **one
forward pass gives 64 “predict the next character” problems**, each of which sees only the characters before it, and
all 64 contribute to the loss. This is why a GPT can be trained in parallel, while the RNN of Lesson 15 has to go
one step at a time.

The optimizer is AdamW from Lesson 07, with the coefficients of GPT-3 and Llama: $\beta_1 = 0.9$, $\beta_2 = 0.95$,
weight decay $\lambda = 0.1$ on the $W$ matrices only, not on the embeddings, biases and LayerNorm. The learning rate
rises linearly over the first $5\%$ of the steps (warmup), then decays along a cosine down to $0.1\eta$. The learning
rate $\eta = 0.04$ was chosen from $\{0.02, 0.03, 0.04, 0.06\}$ by validation loss, with the very `train` function
below (section 13 has the sweep ready for you to rerun). Because $\eta$ was chosen on the validation set and the
notebook has no separate test set, the final number is slightly optimistic.

In [ ]:
def val_bits(p, cfg):
    """Bits/character on the 8,000 validation characters, cut into consecutive windows of 64 characters."""
    seq = np.concatenate([train_ids[-1:], val_ids])  # the first character of the validation set is predicted too
    n = (len(seq) - 1) // cfg["ctx"]
    X = seq[: n * cfg["ctx"]].reshape(n, -1)
    Y = seq[1 : n * cfg["ctx"] + 1].reshape(n, -1)
    return cross_entropy(forward(p, X, cfg)[0], Y)[0] / math.log(2)


def lr_at(it, steps, eta):
    """Linear warmup over the first 5% of the steps, then a cosine from η down to 0.1·η."""
    warm = steps // 20
    if it <= warm:
        return eta * it / warm
    return eta * (0.1 + 0.45 * (1 + math.cos(math.pi * (it - warm) / (steps - warm))))


def adamw_step(p, g, state, lr, it, b1=0.9, b2=0.95, wd=0.1):
    """AdamW (Lesson 07): decoupled weight decay, on the W matrices only."""
    for k_ in p:
        m, s = state.setdefault(k_, (np.zeros_like(p[k_]), np.zeros_like(p[k_])))
        m[:] = b1 * m + (1 - b1) * g[k_]
        s[:] = b2 * s + (1 - b2) * g[k_] * g[k_]
        if k_.split(".")[-1].startswith("W"):
            p[k_] *= 1 - lr * wd
        p[k_] -= lr * (m / (1 - b1**it)) / (np.sqrt(s / (1 - b2**it)) + 1e-8)


def train(cfg, steps=650, eta=0.04, B=16, seed=0, eval_every=65):
    p, state, rng = init_gpt(cfg, seed), {}, np.random.default_rng(seed)
    n = cfg["ctx"]
    log = {"step": [], "loss": [], "val_step": [0], "val": [val_bits(p, cfg)]}
    for it in range(1, steps + 1):
        start = rng.integers(0, len(train_ids) - n - 1, B)
        W = train_ids[start[:, None] + np.arange(n + 1)]  # 16 windows of 65 characters each
        z, cache = forward(p, W[:, :-1], cfg)
        loss, dz = cross_entropy(z, W[:, 1:])  # 16 × 64 next-character problems at once
        adamw_step(p, backward(p, cache, dz, cfg), state, lr_at(it, steps, eta), it)
        log["step"].append(it)
        log["loss"].append(loss / math.log(2))
        if it % eval_every == 0 or it == steps:
            log["val_step"].append(it)
            log["val"].append(val_bits(p, cfg))
    return p, log


t0 = time.perf_counter()
params, log = train(CFG)
print(
    f"650 steps × 16 windows × 64 characters = {num(650 * 16 * 64, 0)} predictions ({num(time.perf_counter() - t0, 1)} s)"
)
gpt_bits = log["val"][-1]
print(
    f"validation: {num(gpt_bits, 3)} bits/character · bigram {num(bigram_bits, 3)} · uniform guess {num(uniform_bits, 3)}"
)
# a wide tolerance: float32 on a different CPU (Colab is x86) makes the training trajectory drift gradually
assert abs(gpt_bits - 2.557) < 0.05, "differs from the reference run: check it again"

The figure below plots the loss of each training batch (light gray), its 25-step moving average (blue, solid line)
and the validation loss every 65 steps (orange, squares). The two horizontal lines are the two baselines of
section 2: uniform guess and bigram.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
loss = np.array(log["loss"])
smooth = np.convolve(loss, np.ones(25) / 25, mode="valid")
ax.plot(log["step"], loss, color=GREY, lw=0.6, alpha=0.5)
ax.plot(log["step"][24:], smooth, color=BLUE, lw=2, label="training (25-step average)")
ax.plot(log["val_step"], log["val"], color=ORANGE, marker="s", ls="--", lw=1.5, label="validation")
for y, name in ((uniform_bits, "uniform guess"), (bigram_bits, "bigram")):
    ax.axhline(y, color="black", lw=0.8, ls=":")
    ax.text(650, y + 0.08, name, ha="right", fontsize=9)
ax.set(xlabel="step", ylabel="bits per character", ylim=(2, 7.4), xlim=(0, 650))
ax.set_title("A small GPT (72,516 parameters) learns The Tale of Kiều in 650 steps", fontsize=11)
ax.legend(frameon=False, loc="upper right", bbox_to_anchor=(1, 0.9))
fmt_axes(fig)
plt.show()
first_below = log["val_step"][next(i for i, b in enumerate(log["val"]) if b < bigram_bits)]
print(f"validation drops below the bigram from step {first_below} on; final {num(gpt_bits, 3)} bits/character")
LSTM_15 = {"params": 150660, "steps": 4000, "bits": 2.2637}  # Lesson 15's demo (rnn-kieu.json): LSTM H = 128, seed 0
print(
    f"for comparison, Lesson 15's LSTM: {num(LSTM_15['params'], 0)} parameters, best within {num(LSTM_15['steps'], 0)} steps,"
    f" {num(LSTM_15['bits'], 2)} bits/character"
)

The loss starts at 7.04 bits (uniform guess), passes the bigram baseline after about 200 steps, and ends at 2.56
bits after 650 steps. The training and validation losses stay close together: a model with 72,516 parameters hasn't
had time to memorize 96,204 characters. For comparison: Lesson 15's LSTM (the number from Lesson 15's demo, printed
above) reaches 2.26 bits on the same 254 validation lines, measured in a slightly different way: the LSTM reads the
text as one continuous stream, so every character has plenty of context, while here the first character of each
64-character window has almost no context. The small GPT gets only 650 steps of practice so that the whole notebook
runs in under a minute; section 13 suggests a few ways to make it better.

## 8 · The mask really hides the future

We redo the test the demo runs on GPT-2: run some verse through the model, then **change one character in the
middle** and run it again.

### 🤔 Predict first
Change the 20th character (index 19) of the text. How much do the predictions at the 19 positions before it change?
And at the positions after it?

In [ ]:
line = "Trăm năm trong cõi người ta,\nChữ tài chữ mệnh khéo là ghét nhau."  # 64 characters: exactly one window
x = np.array([stoi[c] for c in line])
x2 = x.copy()
x2[19] = stoi["x"]  # "người" (person) → "xgười"
print(repr(decode(x[:24])), "→", repr(decode(x2[:24])))
for causal in (True, False):
    za = forward(params, x[None], CFG, causal)[0][0]
    zb = forward(params, x2[None], CFG, causal)[0][0]
    delta = np.abs(zb - za).max(-1)  # the largest change in the logits at each position
    name = "masked  " if causal else "unmasked"
    print(
        f"{name}: positions 0–18 change by at most {sci(delta[:19].max())} · positions 19–{len(x) - 1} by at most {num(delta[19:].max())}"
    )
    if causal:
        assert delta[:19].max() == 0.0 and delta[19:].max() > 0.1

With the mask, the predictions at the 19 positions before the edit **don't change by a single bit** (the difference
is exactly 0, not just small): they never read the 20th character. Remove the mask, with the same weights, and
editing one later character changes the earlier predictions too. The mask is what makes it possible to train all 64
positions in one pass without any position seeing its own answer ahead of time; that is why it is used *in training
too*, not only when generating text. The demo runs exactly this test on GPT-2: change “ largest” to “ oldest”, and
the 19 distributions before it stay the same (section 12).

## 9 · Text generation: temperature, top-k and the KV cache

Training is parallel, but text generation is still **one token at a time**: predict the distribution of the next
token, sample a token, append it to the sequence, repeat. At the sampling step, the logits $z$ are divided by the
**temperature** $T$ before the softmax:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

$T < 1$ makes the distribution sharper, $T > 1$ flatter, and $T \to 0$ means always picking the token with the
largest logit (greedy decoding). **Top-k** keeps only the $k$ tokens with the largest logits, then renormalizes.
Dividing by a positive number doesn't change the order of the logits, so $T$ never changes the top token; it only
changes the gaps between the tokens.

**The KV cache.** Generating token $t+1$ by rerunning the whole sequence is wasteful: the keys and values of the
earlier positions don't change (the mask guarantees that an earlier position never looks at a new token). So we
compute the whole prompt once (*prefill*), store the $K$ and $V$ of every block, and then at each step compute only
**one** new position: its query is scored against every stored key, with no mask needed.

In [ ]:
def prefill(p, prompt, cfg):
    """One masked pass over the whole prompt: the logits at the last position, plus K, V of every block (head, position, d_h)."""
    z, cache = forward(p, np.array(prompt)[None], cfg)
    return z[0, -1], [c["k"][0] for c in cache["layers"]], [c["v"][0] for c in cache["layers"]]


def decode_step(p, token, pos, K, Vc, cfg):
    """One new token at position pos: compute its own q, k, v, append k, v to the cache, attend to every position so far."""
    d, nh = cfg["d"], cfg["heads"]
    h = p["wte"][token] + p["wpe"][pos]  # (d,)
    for l in range(cfg["L"]):
        a, _ = layernorm(h, p[f"{l}.g1"], p[f"{l}.b1"])
        q, k, v = (a @ p[f"{l}.Wqkv"] + p[f"{l}.bqkv"]).reshape(3, nh, d // nh)
        K[l] = np.concatenate([K[l], k[:, None]], axis=1)  # store the new position's key and value
        Vc[l] = np.concatenate([Vc[l], v[:, None]], axis=1)
        o, _ = attention(q[:, None], K[l], Vc[l], causal=False)  # one query, every key so far: no mask needed
        h = h + o.reshape(d) @ p[f"{l}.Wo"] + p[f"{l}.bo"]
        m, _ = layernorm(h, p[f"{l}.g2"], p[f"{l}.b2"])
        h = h + gelu(m @ p[f"{l}.W1"] + p[f"{l}.c1"])[0] @ p[f"{l}.W2"] + p[f"{l}.c2"]
    hf, _ = layernorm(h, p["gf"], p["bf"])
    return hf @ p["Wh"] + p["bh"]

### 🤔 Predict first
Generate 54 characters greedily after the 10-character prompt “Trăm năm t”, once with the KV cache and once by
rerunning the whole sequence. Do the two ways give the same verse? How far apart are the logits?

In [ ]:
prompt = [stoi[c] for c in "Trăm năm t"]
z, K, Vc = prefill(params, prompt, CFG)
seq, cached = list(prompt), [z]
t0 = time.perf_counter()
while len(seq) < CFG["ctx"]:
    seq.append(int(np.argmax(cached[-1])))  # greedy
    if len(seq) < CFG["ctx"]:
        cached.append(decode_step(params, seq[-1], len(seq) - 1, K, Vc, CFG))
t_cache = time.perf_counter() - t0
t0 = time.perf_counter()
full = [forward(params, np.array(seq[:t])[None], CFG)[0][0, -1] for t in range(len(prompt), len(seq))]  # rerun
t_full = time.perf_counter() - t0
gap = np.abs(np.array(cached) - np.array(full)).max()
greedy_full = [int(np.argmax(f)) for f in full]
print(repr(decode(seq)))
print(
    f"same {len(seq) - len(prompt)} characters: {greedy_full == seq[len(prompt) :]} · logits differ by at most {sci(gap)}"
)
print(
    f"(time on this machine: with the cache {num(t_cache * 1000, 0)} ms, rerunning the whole sequence {num(t_full * 1000, 0)} ms)"
)
assert greedy_full == seq[len(prompt) :] and gap < 1e-4

The same 54 characters, and the logits differ only at float32's rounding level: the KV cache is **not an
approximation**; it computes exactly the numbers of the full pass, it just doesn't recompute what it already has. The
price is memory: every token read so far keeps a key and a value in every block (section 10 counts it for GPT-2).

The greedy verse also shows a familiar flaw: always picking the most likely character makes the model repeat itself
(in the reference run: “thôi thôi … thôi”, “thế thế”). That is why we *sample at random*. The cell below generates
1,000 characters with each sampling method, after the last 4 lines of the training set, then measures two things: the
share of lines with exactly 6 or 8 syllables (the lục bát meter, whose lines alternate between 6 and 8 syllables;
Vietnamese writes each syllable as a separate word), and the share of syllables that are real, that is, that occur
in the training set. To sample the next character, the model reads only the last 64 characters (its whole context
window).

In [ ]:
SYLLABLES = {w for ln in LINES[:3000] for w in re.sub(r"[^\w\s]", " ", ln.lower()).split()}


def generate(p, prompt, n, cfg, T=1.0, k=0, seed=0):
    """n tokens after the prompt; T = 0 is greedy, k > 0 is top-k. Each step reads only the last cfg["ctx"] tokens."""
    rng = np.random.default_rng(seed)
    seq = list(prompt)
    for _ in range(n):
        z = forward(p, np.array(seq[-cfg["ctx"] :])[None], cfg)[0][0, -1].astype(np.float64)
        if T == 0:
            seq.append(int(np.argmax(z)))
            continue
        keep = np.argsort(-z, kind="stable")[:k] if k else np.arange(len(z))  # top-k: keep only the k largest logits
        seq.append(int(keep[rng.choice(len(keep), p=softmax(z[keep] / T))]))
    return seq[len(prompt) :]


def poem_stats(s):
    """The share of lines with 6 or 8 syllables, and of syllables found in the training set (minus the last line, which may be cut off)."""
    words = [re.sub(r"[^\w\s]", " ", ln.lower()).split() for ln in s.split("\n")[:-1]]
    flat_words = [w for ws in words for w in ws]
    return len(words), np.mean([len(ws) in (6, 8) for ws in words]), np.mean([w in SYLLABLES for w in flat_words])


cue = [stoi[c] for c in "\n".join(LINES[2996:3000]) + "\n"]
N_GEN = 1000
print(f"each sampling method generates {num(N_GEN, 0)} characters")
print(
    f"real verse (254 validation lines): {pct(poem_stats(chr(10).join(LINES[3000:]) + chr(10))[1], 0)} lines of 6/8 syllables"
)
rng = np.random.default_rng(0)
bi = [stoi["\n"]]
for _ in range(N_GEN):  # bigram, T = 1: looks only at the preceding character
    bi.append(int(rng.choice(V, p=P_bigram[bi[-1]])))
for name, s in [("bigram, T = 1", decode(bi[1:]))] + [
    (f"small GPT, T = {num(T, 1)}" + (f", top-k {k}" if k else ""), decode(generate(params, cue, N_GEN, CFG, T, k)))
    for T, k in ((0.5, 0), (1.0, 0), (1.0, 5))
]:
    n_lines, share68, real = poem_stats(s)
    print(f"{name:<22} {n_lines:>3} lines · {pct(share68, 0):>4} with 6/8 syllables · {pct(real, 0):>4} real syllables")
    print("   ", " / ".join(s.split("\n")[:3]))

Each row prints the statistics of 1,000 characters, with the first 3 lines. The bigram model produces words that
don't exist, and lines of arbitrary length. The small GPT has learned far more: a capital letter at the start of a
line, lines ending in a comma or a period, line lengths close to the lục bát meter, and mostly real syllables, even
though the verse doesn't mean anything yet. A low temperature ($T = 0.5$) gives the most real syllables; $T = 1$
also samples unlikely characters, so more syllables are misspelled. Top-k 5 keeps $T = 1$ but cuts off the tail of
the distribution, and the share of real syllables rises again. Each number comes from a single sampling run with
seed 0, so a difference of a few percent means nothing yet; what is worth looking at is their direction.

## 10 · The cost of GPT-2: parameters, compute, memory

The small GPT has 72,516 parameters. GPT-2 small is the same architecture with its published configuration: 12
blocks, $d = 768$, 12 heads × 64 dimensions, an MLP $768 \to 3{,}072 \to 768$, a vocabulary of 50,257 BPE tokens
(Lesson 14), a window of 1,024 tokens, and an output layer that reuses $E^\top$ (no separate $W_h$). The cell below
counts the parameters part by part and compares them with the count on the real weights from the demo.

In [ ]:
GPT2 = {"L": 12, "d": 768, "heads": 12, "ff": 3072, "V": 50257, "ctx": 1024}  # OpenAI's hparams.json


def gpt2_params(c):
    d, ff, L = c["d"], c["ff"], c["L"]
    return {
        "E: token embedding": c["V"] * d,
        "P: position embedding": c["ctx"] * d,
        "attention: W_QKV, b_QKV": L * (d * 3 * d + 3 * d),
        "attention: W_O, b_O": L * (d * d + d),
        "MLP: W_1, c_1": L * (d * ff + ff),
        "MLP: W_2, c_2": L * (ff * d + d),
        "LayerNorm: g, b": (2 * L + 1) * 2 * d,
        "output: reuses Eᵀ": 0,
    }


parts = gpt2_params(GPT2)
total = sum(parts.values())
for name, n_ in parts.items():
    print(f"{name:<24} {num(n_, 0):>11}  {pct(n_ / total, 1):>6}")
print(f"{'total':<24} {num(total, 0):>11}")
attn, mlp = (
    parts["attention: W_QKV, b_QKV"] + parts["attention: W_O, b_O"],
    parts["MLP: W_1, c_1"] + parts["MLP: W_2, c_2"],
)
block_ln = GPT2["L"] * 2 * 2 * GPT2["d"]  # two LayerNorms per block
print(
    f"within the blocks, the MLP holds {pct(mlp / (attn + mlp + block_ln), 1)} of the parameters, attention {pct(attn / (attn + mlp + block_ln), 1)}"
)
real = inside["params"]  # counted on the real weight arrays (demo)
print(
    f"counted on the real weights: {num(real['total'], 0)} · attention {num(real['attn'], 0)} · MLP {num(real['mlp'], 0)}"
)
assert total == 124439808 and (attn, mlp, parts["E: token embedding"]) == (real["attn"], real["mlp"], real["wte"])

Exactly 124,439,808, the same as the count on the real weight arrays. (The GPT-2 paper says 117 million; OpenAI's
README later corrected this, saying the original numbers were miscounted, and the small model is now called 124M.)
The embedding table takes up almost a third of the model, because the vocabulary is large; within the blocks, the
MLP holds about 2/3 of the parameters, just as the lesson page says.

**Compute.** Following the demo's convention, a $d_\text{in} \times d_\text{out}$ matrix multiplication costs
$d_\text{in}\,d_\text{out}$ multiply-accumulate (MAC) operations per position; biases, LayerNorm and softmax are not
counted. Each position goes through the matrices of the 12 blocks (a fixed amount), attends to $t$ positions
($q \cdot k$, then summing the $v$: $2 \cdot 12 \cdot 768 \cdot t$ MACs), and the output layer is needed only for
the generated tokens. The cell below writes the formula for generating $N$ tokens after a $P$-token prompt, with
and without the cache, and checks it by recounting position by position.

In [ ]:
def dense_mac(c):  # Q, K, V, W_O and the MLP of every block, for one position
    return c["L"] * (c["d"] * 3 * c["d"] + c["d"] * c["d"] + 2 * c["d"] * c["ff"])


def attn_mac(c, t):  # one position attending to t positions (itself included)
    return 2 * c["L"] * c["d"] * t


def out_mac(c):
    return c["d"] * c["V"]


def gen_mac(c, P, N, cache):
    """Generate N tokens after a P-token prompt. With the cache: every position goes through the model exactly once (the
    last generated token doesn't need to be fed back in). Without it: step i reruns all P + i positions."""
    tri = attn_mac(c, 1)  # Σ_{t=1}^{n} attn_mac(t) = tri · n(n + 1)/2
    if cache:
        n = P + N - 1
        return n * dense_mac(c) + tri * n * (n + 1) // 2 + N * out_mac(c)
    return sum((P + i) * dense_mac(c) + tri * (P + i) * (P + i + 1) // 2 for i in range(N)) + N * out_mac(c)


def recount(c, P, N, cache):  # recount every position processed
    passes = [range(1, P + N)] if cache else [range(1, P + i + 1) for i in range(N)]
    return sum(dense_mac(c) + attn_mac(c, t) for r in passes for t in r) + N * out_mac(c)


print(f"per position: {num(dense_mac(GPT2), 0)} MACs in the blocks + {num(out_mac(GPT2), 0)} MACs in the output layer")
print(
    f"   = {num((dense_mac(GPT2) + out_mac(GPT2)) / 1e6, 1)} million MACs for one new token (while the text is short);"
    f" attention at position 1,024 adds {num(attn_mac(GPT2, 1024) / 1e6, 1)} million"
)
for P, N in ((6, 30), (6, 1018)):
    w, wo = gen_mac(GPT2, P, N, True), gen_mac(GPT2, P, N, False)
    assert w == recount(GPT2, P, N, True) and wo == recount(GPT2, P, N, False)
    print(
        f"prompt {P} + {num(N, 0)} generated tokens: with the cache {num(w / 1e9, 1)} billion MACs, rerunning {num(wo / 1e9, 1)} billion"
        f" ({num(wo / w, 0)} times as many)"
    )
kv_bytes = 2 * GPT2["L"] * GPT2["d"] * 2  # one key and one value, 12 blocks × 768 numbers, fp16 = 2 bytes
print(
    f"KV cache: {num(kv_bytes / 1024, 0)} KiB per token (fp16), {num(kv_bytes * 1024 / 2**20, 0)} MiB for 1,024 tokens"
)
print(f"our small GPT: {2 * CFG['L'] * CFG['d'] * 4} bytes per token (float32)")
assert (dense_mac(GPT2), out_mac(GPT2)) == (84934656, 38597376) and kv_bytes == 36864
assert (gen_mac(GPT2, 6, 30, True), gen_mac(GPT2, 6, 30, False)) == (4142246400, 53535306240)

These are exactly the numbers the demo shows for the reel's 6-token prompt. While the text is short, each new token
costs about 123.5 million MACs, roughly **one MAC per parameter** (Kaplan et al., 2020, write this as $\approx 2N$
FLOPs per token). The attention part grows with the length: at position 1,024 it adds
$2 \cdot 12 \cdot 768 \cdot 1{,}024$ MACs, about 18.9 million. Without the cache, the total work grows with the
square of the number of generated tokens: the figure below (log scale on both axes) plots the total MACs to generate
$N$ tokens after a 6-token prompt, with the cache (blue, round dots) and rerunning the whole sequence (orange,
squares, dashed line).

In [ ]:
Ns = np.unique(np.geomspace(1, 1018, 30).astype(int))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(Ns, [gen_mac(GPT2, 6, N, True) for N in Ns], color=BLUE, marker="o", ms=4, label="with the KV cache")
ax.plot(
    Ns,
    [gen_mac(GPT2, 6, N, False) for N in Ns],
    color=ORANGE,
    marker="s",
    ms=4,
    ls="--",
    label="recomputing the whole sequence",
)
ax.set(xscale="log", yscale="log", xlabel="number of generated tokens N", ylabel="total MACs")
ax.set_title("GPT-2 small, 6-token prompt: multiply-accumulates to generate N tokens", fontsize=11)
ax.legend(frameon=False)
ax.grid(alpha=0.3, which="major")
fmt_axes(fig)
plt.show()

The two lines start out the same, then pull far apart: for $N$ = 1,018 tokens, rerunning the whole sequence costs
352 times as much. The KV cache trades compute for memory. GPT-2 needs 36 KiB per token (fp16) for each sequence
being generated, multiplied by the number of sequences served at the same time. For larger models, it is this memory
that limits how many users one GPU can serve (section 14).

## 11 · Induction heads: why at least two blocks are needed

After training, many of GPT-2's heads do jobs that are easy to name (section 12). The most famous is **copying**: on
seeing token [A] again, look at what came after [A] last time ([B]), then predict [B]. Olsson et al. (2022) call a
head that does this an **induction head**. It needs two heads in two consecutive blocks:
1. block 0 has a **previous-token head**, which copies “the token before me is [A]” into the vector of the position
   holding [B];
2. in block 1, the query of the current position (token [A]) looks for the position that records “before me is
   [A]”, which is exactly the position of [B], and copies [B] out.

We test this on a pure task: sequences of 25 random symbols (16 kinds), in which a segment 3–8 symbols long is
repeated right after itself. The loss counts only the symbols of the repeat that the earlier segment gives away (from
the second symbol of the repeat on). Random guessing costs $\log_2 16 = 4$ bits. A small model: $d = 32$, 2 heads,
a window of 24 positions; one version with 2 blocks and one with 1 block, using the same `forward`, `backward` and
AdamW as above.

In [ ]:
def copy_batch(rng, B, n_pos=24, n_sym=16):
    """Random sequences, each with a segment 3–8 long repeated right after it. w = 1 where the earlier segment gives away the answer."""
    X = rng.integers(0, n_sym, (B, n_pos + 1))
    w = np.zeros((B, n_pos))
    seg = []
    for b in range(B):
        n_ = rng.integers(3, 9)
        s0 = rng.integers(0, n_pos + 1 - 2 * n_)
        X[b, s0 + n_ : s0 + 2 * n_] = X[b, s0 : s0 + n_]  # the repeat
        w[b, s0 + n_ : s0 + 2 * n_ - 1] = 1  # predict symbols 2…n of the repeat
        seg.append((s0, n_))
    return X, w, seg


def train_copy(L, steps=600, eta=0.01, seed=0):
    cfg = {"V": 16, "ctx": 24, "d": 32, "L": L, "heads": 2}
    p, state, rng = init_gpt(cfg, seed), {}, np.random.default_rng(seed)
    Xe, we, _ = copy_batch(np.random.default_rng(99), 256)  # a fixed test batch
    curve = []
    for it in range(1, steps + 1):
        X, w, _ = copy_batch(rng, 32)
        z, cache = forward(p, X[:, :-1], cfg)
        _, dz = cross_entropy(z, X[:, 1:], w)
        adamw_step(p, backward(p, cache, dz, cfg), state, lr_at(it, steps, eta), it)
        if it % 50 == 0:
            curve.append(cross_entropy(forward(p, Xe[:, :-1], cfg)[0], Xe[:, 1:], we)[0] / math.log(2))
    return p, cfg, curve

### 🤔 Predict first
Both models are trained the same way for 600 steps. Can the **one-block** model, with its 2 attention heads, learn
to copy?

In [ ]:
t0 = time.perf_counter()
copy_runs = {L: train_copy(L) for L in (1, 2)}
for L, (_, _, curve) in copy_runs.items():
    print(f"{L}-block model: the repeat costs {num(curve[0], 2)} → {num(curve[-1], 2)} bits/symbol (random guess: 4)")
print(f"({num(time.perf_counter() - t0, 1)} s)")
assert copy_runs[2][2][-1] < 1.5 and copy_runs[1][2][-1] > 2.0, "differs from the reference run"

fig, ax = plt.subplots(figsize=(7, 3.4))
steps_ = np.arange(50, 601, 50)
ax.plot(steps_, copy_runs[1][2], color=ORANGE, marker="s", ms=4, ls="--", label="1 block")
ax.plot(steps_, copy_runs[2][2], color=BLUE, marker="o", ms=4, label="2 blocks")
ax.axhline(4, color="black", lw=0.8, ls=":")
ax.text(600, 4.05, "random guess", ha="right", va="bottom", fontsize=9)
ax.set(xlabel="step", ylabel="bits per symbol of the repeat", ylim=(0, 4.4), xlim=(0, 600))
ax.set_title("Copying the repeat: 1-block and 2-block models", fontsize=11)
ax.legend(frameon=False, loc="lower left")
fmt_axes(fig)
plt.show()

The two-block model learns to copy: the repeat costs about 1 bit per symbol or less, against the 4 bits of random
guessing. The one-block model stalls at about 2.5 bits, better than random guessing, but it cannot copy. One block
only knows “which symbols *have appeared*” (it can look back at every earlier symbol and raise their probability),
but not which symbol came *after* the previous occurrence of the current token: the information “what comes before
me” isn't in any position's vector yet, because an attention layer has to put it there first.

The cell below opens up the two-block model. For each head, it measures two scores on the test batch: the
**previous-token score** (the mean of $\alpha_{t,t-1}$) and the **induction score** (the mean attention that a
position of the repeat puts on exactly the position holding the answer in the original segment). Then it draws the
attention matrix of one sequence: on the left the previous-token head of block 0, on the right the induction head of
block 1. Dark means a large weight; white circles mark the entries each kind of head “should” look at.

In [ ]:
p_copy, cfg_copy, _ = copy_runs[2]
Xe, _, seg = copy_batch(np.random.default_rng(99), 256)
_, cache = forward(p_copy, Xe[:, :-1], cfg_copy)
prev_score, ind_score = [], []
for c in cache["layers"]:
    A_ = c["A"]  # (batch, head, 24, 24)
    prev_score.append(A_[:, :, np.arange(1, 24), np.arange(0, 23)].mean((0, 2)))
    hits = [A_[b, :, t, t - n_ + 1] for b, (s0, n_) in enumerate(seg) for t in range(s0 + n_, s0 + 2 * n_ - 1)]
    ind_score.append(np.mean(hits, axis=0))
for l in range(2):
    print(
        f"block {l}: previous-token score {' / '.join(num(x, 2) for x in prev_score[l])} (head 0 / head 1)"
        f" · induction score {' / '.join(num(x, 2) for x in ind_score[l])}"
    )
h_prev, h_ind = int(np.argmax(prev_score[0])), int(np.argmax(ind_score[1]))
assert prev_score[0][h_prev] > 0.25 and ind_score[1][h_ind] > 0.4 > ind_score[0].max()

b = next(i for i, (s0, n_) in enumerate(seg) if n_ >= 6)  # a sequence with a long repeated segment
s0, n_ = seg[b]
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.4))
for ax, l, hd, title in (
    (axes[0], 0, h_prev, "block 0, head {}: previous-token head"),
    (axes[1], 1, h_ind, "block 1, head {}: induction head"),
):
    ax.imshow(cache["layers"][l]["A"][b, hd], cmap=SEQ, vmin=0, vmax=1)
    want = [(t, t - 1) for t in range(1, 24)] if l == 0 else [(t, t - n_ + 1) for t in range(s0 + n_, s0 + 2 * n_ - 1)]
    ax.scatter([j for _, j in want], [t for t, _ in want], s=70, facecolors="none", edgecolors="white", lw=1.2)
    ax.set(xlabel="attended position j", ylabel="query position t", title=title.format(hd))
    ax.set_xticks(range(0, 24, 4))
    ax.set_yticks(range(0, 24, 4))
fig.suptitle(
    f"A sequence with a segment of {n_} symbols at positions {s0}–{s0 + n_ - 1}, repeated at {s0 + n_}–{s0 + 2 * n_ - 1}",
    fontsize=11,
)
plt.tight_layout()
fmt_axes(fig)
plt.show()

The scores, averaged over the test batch, are just as in the diagram. In block 0, both heads look at the previous token
(about a third of their attention, many times the average level): in the figure, the dark cells lie just below the
diagonal. In block 1, the two heads put about 0.6 of their total attention weight on exactly the position holding the
answer in the original segment: within the repeat, the dark cells run parallel to the diagonal, one less than the
repeat length away from it (the cell $(t, t - n + 1)$ for a segment of length $n$). A single sequence shows the pattern
only in part: in the one in the figure, many of block 1's circled cells are still pale, and in block 0's last rows the
head piles its attention onto one column instead of the previous token. Nobody programmed these two jobs; they emerged
on their own because they help lower the loss. GPT-2 has the same circuit, at a larger scale (section 12).

## 12 · Inside the real GPT-2

From here on we read the measurements on GPT-2 small that the demo uses (`inside`, `samp`, loaded in section 2).
GPT-2's tokens are byte-level BPE tokens (Lesson 14). A token that starts with a space (“ Paris”) begins a new word.
A Vietnamese word with diacritics is often cut into several UTF-8 byte fragments; the notebook prints the fragments
that are not a complete character as hex codes in square brackets.

**One pass, every position.** As with the small GPT in section 7, one masked pass over the sentence below gives a
prediction at every position. The cell below prints, at each position, the token GPT-2 considers most likely, along
with the probability and the rank it gives the actual next token.

In [ ]:
TOK = {}
for tokens in (inside["tokens"], samp["tokens"]):
    TOK |= {int(i): s for i, s in tokens["text"].items()} | {int(i): f"[{h}]" for i, h in tokens["raw"].items()}

r_en, r_vi = inside["read"]["en"], inside["read"]["vi"]
toks = [TOK[i] for i in r_en["ids"]]
print(r_en["text"])
for t, pos in enumerate(r_en["pos"]):
    (best, p_best), hit = pos["top"][0], "  ← correct" if pos["rank"] == 1 else ""
    print(
        f"{t:>2} after “{toks[t]}”: predicts “{TOK[best]}” {pct(p_best, 1):>6} · actual token “{toks[t + 1]}” rank"
        f" {num(pos['rank'], 0)}, {pct(pos['p'], 2)}{hit}"
    )
hits = sum(pos["rank"] == 1 for pos in r_en["pos"])
print(f"rank 1 at {hits}/{len(r_en['pos'])} positions · mean {num(r_en['bitsPerToken'], 2)} bits/token")

One pass gives all 27 predictions. There are places where GPT-2 is almost certain (“ Asia” after “ Southeast”,
“ City” after “ Ho Chi Minh”), and places it cannot know in advance (the first token, or “ is” after “Vietnam”). At
the first position, the model has exactly one token to read.

Now the test of section 8, on GPT-2: change “ largest” to “ oldest”. The cell below prints the largest change in the
logits at each position, and compares the English sentence with its Vietnamese translation.

In [ ]:
e = inside["read"]["en"]["edit"]
print(f"change “{TOK[e['from']]}” → “{TOK[e['to']]}” at token {e['pos']}:")
print(f"  positions 0–{e['pos'] - 1}: largest Δ logit {num(max(e['delta'][: e['pos']]), 3)}")
print(f"  positions {e['pos']}–{len(e['delta']) - 1}: {' · '.join(num(x, 1) for x in e['delta'][e['pos'] :])}")
print(f"  after “ oldest”, GPT-2 predicts: {', '.join(f'“{TOK[i]}” {pct(q, 1)}' for i, q in e['top'][0])}")
for name, r_ in (("English", r_en), ("Vietnamese", r_vi)):
    print(
        f"{name:<10}: {r_['bytes']} bytes → {len(r_['ids'])} tokens · {num(r_['bitsPerToken'], 2)} bits/token ·"
        f" {num(r_['bitsPerByte'], 2)} bits/byte"
    )
print("first 10 tokens of the Vietnamese sentence:", " | ".join(TOK[i] for i in r_vi["ids"][:10]))
for lang, fl in inside["read"]["flores"].items():
    print(
        f"FLORES-200 ({lang}, {fl['sentences']} sentences): {num(fl['bitsPerByte'], 2)} bits/byte, {num(fl['bytes'] / fl['tokens'], 2)} bytes/token"
    )

With the mask, the 19 distributions before “ oldest” stay absolutely unchanged (Δ = 0), just like the small GPT in
section 8. From the edited position on, the logits change by up to about 10.

The Vietnamese sentence with the same meaning is more than 3 times as long in tokens, and costs more than 4 times as
many bits per byte. GPT-2's BPE tokenizer learned from English text, so it cuts many words with diacritics into
single UTF-8 bytes (“Việt” into “Vi”, the three byte fragments of “ệ”, then “t”). The GPT-2 paper also states that
they deliberately filtered non-English pages out of WebText. The same holds on 40 FLORES-200 sentences per language:
each Vietnamese token holds only 1.55 bytes (English: 4.70), and each byte costs almost 3 times as many bits.

### The logit lens: reading the residual stream after each block

The output layer reads only the final vector of the residual stream. But we can try reading it *early*: take the
vector at the last position after block $\ell$, pass it through the final LayerNorm and multiply by $E^\top$, as if
the model stopped there (nostalgebraist, 2020). The cell below does this for three prompts and prints the rank of
the answer at each depth (0 is right after the embedding, 12 is the real output).

In [ ]:
def gpt_text(token_ids):
    return "".join(TOK[i] for i in token_ids)


for x in inside["lens"]:
    ranks = [dp["rank"] for dp in x["depth"]]
    first = next(i for i in range(len(ranks)) if all(r == 1 for r in ranks[i:]))
    print(f"“{gpt_text(x['ids'])}” → “{TOK[x['answer']]}”")
    print(f"   rank by depth 0–12: {' · '.join(num(r, 0) for r in ranks)}")
    tops = " → ".join(f"{L}: “{TOK[x['depth'][L]['top'][0][0]]}”" for L in (0, 6, 9, 10, 11, 12))
    print(f"   ranked first from depth {first} on; the top token at depth {tops}")
    print(
        f"   probability at the output {pct(x['p'], 2)}; vector length {num(x['depth'][0]['norm'], 1)} → {num(x['depth'][-1]['norm'], 1)}"
    )

fig, ax = plt.subplots(figsize=(7.6, 3.8))
for x, c, mk, ls in zip(inside["lens"], (BLUE, ORANGE, GREEN), "os^", ("-", "--", ":")):
    ax.plot(range(13), [dp["rank"] for dp in x["depth"]], color=c, marker=mk, ls=ls, label=f"“{TOK[x['answer']]}”")
ax.set(yscale="log", xlabel="number of blocks passed", ylabel="rank of the answer (1 = top)", xlim=(-0.3, 12.3))
ax.invert_yaxis()
ax.set_xticks(range(13))
ax.set_title("The logit lens on GPT-2 small: the answer climbs to the top in the last blocks", fontsize=11)
ax.legend(frameon=False, title="answer", loc="lower right")
fmt_axes(fig)
plt.show()

The three lines (blue solid “ Paris”, orange dashed “ Mary”, green dotted “ visualize”; the vertical axis on a log
scale, rank 1 at the top) tell the same story. Right after the embedding, the answer ranks in the thousands to tens
of thousands. It climbs gradually through the middle blocks and only reaches the top after 10–11 of the 12 blocks.
For the Eiffel Tower sentence, after 10 blocks the vector still reads out “ London”, and only after 11 blocks
“ Paris”. The length of the vector at the last position grows through the blocks: each sublayer *adds* to the
residual stream rather than replacing it.
The logit lens is only an approximate view: the middle blocks are not trained to be read through $E^\top$, and the
*tuned lens* (Belrose et al., 2023) learns a separate projection for each block to read them more accurately.

### Knocking out one sublayer at a time

Another test: set the output of exactly one attention layer or one MLP to 0 (keeping the skip connection) and
measure again. The cell below prints the probability of the answer with each sublayer knocked out, and the extra bits
per token on 10 English FLORES-200 sentences.

In [ ]:
for x in inside["lens"]:
    ab, ans = x["ablate"], TOK[x["answer"]]
    rest = [a for kind in ("attn", "mlp") for a in ab[kind][1:]]  # the 22 sublayers outside block 0
    keep = sum(a["top"] == x["answer"] for a in rest)
    print(
        f"“{ans}” {pct(x['p'], 2)} · without block 0's attention: {pct(ab['attn'][0]['p'], 2)} · without block 0's MLP: {pct(ab['mlp'][0]['p'], 2)}"
        f" · without block 11's MLP: {pct(ab['mlp'][11]['p'], 2)} · without 1 of the {len(rest)} other sublayers: still on top {keep}/{len(rest)}"
    )
AF = inside["ablateFlores"]
print(f"FLORES-en ({AF['sentences']} sentences): {num(AF['base'], 2)} bits/token with every sublayer")
print(f"   without block 0: +{num(AF['attn'][0], 2)} (attention), +{num(AF['mlp'][0], 2)} (MLP)")
worst_kind, worst_l = max(((k_, l) for k_ in ("attn", "mlp") for l in range(1, 12)), key=lambda kl: AF[kl[0]][kl[1]])
print(
    f"   without one other sublayer: at most {num(AF[worst_kind][worst_l], 2)} extra bits"
    f" (block {worst_l}'s {'attention' if worst_kind == 'attn' else 'MLP'})"
)

fig, ax = plt.subplots(figsize=(7.6, 3.4))
ax.plot(range(12), AF["attn"], color=BLUE, marker="o", label="without the attention layer")
ax.plot(range(12), AF["mlp"], color=ORANGE, marker="s", ls="--", label="without the MLP")
ax.set(yscale="log", xlabel="block", ylabel="extra bits/token", xticks=range(12))
ax.set_title("GPT-2 small, English FLORES-200: knocking out one sublayer at a time", fontsize=11)
ax.legend(frameon=False)
fmt_axes(fig)
plt.show()

Knock out the attention or the MLP of **block 0** and the model collapses: each token costs 3–5 extra bits, and the
answer drops below 1%. Knocking out any other sublayer adds at most half a bit, and the answer usually stays on top.
As with ResNet in Lesson 13, the blocks all write into one residual stream, so losing one block doesn't break the
whole network. More surprising: knocking out the last block's MLP actually *raises* the probability of all three
answers (6.38% → 12.25% for “ Paris”), even though on FLORES this very MLP is the most important sublayer outside
block 0. It seems to make the distribution less certain: that helps the average loss on real text, but not these
particular sentences.

### 144 attention heads

GPT-2 small has 12 blocks × 12 heads. The demo measures each head on a passage of *Alice's Adventures in Wonderland*
(Carroll, 1865, public domain), with `<|endoftext|>` in front: the share of attention on the first token and on the
previous token; the induction score is measured as in section 11, on 16 sequences of 25 random tokens repeated twice.
Heads are numbered from 0, as in the papers: “5.5” is block 5, head 5. The cell below draws three 12 × 12 heat maps
(rows are blocks, columns are heads, dark means a high score) and prints the top heads.

In [ ]:
H = inside["heads"]
maps = {"first token": H["first"], "previous token": H["prev"], "induction score": H["induction"]}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (name, M) in zip(axes, maps.items()):
    im = ax.imshow(np.array(M), cmap=SEQ, vmin=0, vmax=1)
    ax.set(title=name, xlabel="head", xticks=range(0, 12, 2), yticks=range(0, 12, 2))
axes[0].set_ylabel("block")
fig.colorbar(im, ax=axes, fraction=0.02)
fig.suptitle("GPT-2 small: each cell is one attention head", fontsize=11)
fmt_axes(fig)
plt.show()
for name, M in maps.items():
    M = np.array(M)
    order = np.argsort(-M.ravel(), kind="stable")[:4]
    print(f"{name:<17}: " + " · ".join(f"{i // 12}.{i % 12} {pct(M.ravel()[i], 1)}" for i in order))
first = np.array(H["first"])
print(f"{(first > 0.5).sum()}/144 heads put more than half their attention on the first token (attention sinks)")
print(f"mean by block: {' · '.join(pct(x, 0) for x in first.mean(1))}")

Three groups of heads stand out clearly, and none of them was assigned its job in advance:
- **Attention sinks** (Xiao et al., 2023): about half of the heads, especially in the middle and last blocks, put
  most of their attention on the first token. Softmax must always hand out the full 100%, so when a head has nothing
  to fetch, it needs a harmless place to “park” its attention, and the first token (visible from every position)
  becomes that place.
- **Previous-token heads:** head 4.11 puts almost all its attention on the previous token, a nearly perfect shift by
  one position, like the block-0 head of the tiny copying model in section 11.
- **Induction heads:** 5.5, 7.10, 6.9, 5.1 sit in blocks after the previous-token head, exactly the order the
  two-step circuit requires.

The repeated-sequence test shows what they do:

In [ ]:
R = H["repeat"]
print(
    f"{R['seqs']} sequences of {R['n']} random tokens, repeated twice (uniform guess over 50,257 tokens: {num(math.log2(50257), 2)} bits)"
)
print(f"   first copy: {num(R['first'], 2)} bits/token, correct at rank 1 {pct(R['top1First'], 1)}")
print(f"   second copy: {num(R['second'], 2)} bits/token, correct at rank 1 {pct(R['top1Second'], 1)}")
for g in R["groups"]:
    names = ", ".join(f"{a}.{b}" for a, b in g["induction"])
    print(
        f"   without the {g['k']:>2} strongest induction heads ({names}): second copy {num(g['bitsInduction'], 2)} bits"
        f" · without {g['k']} random heads: {num(g['bitsRandom'], 2)}"
    )

The first time through, the random sequence costs almost 19 bits per token, *worse* than even the uniform guess
(15.62 bits), because GPT-2 expects meaningful text. The repeat costs only about half a bit and is correct at rank 1
at 99% of the positions: the model copies from the earlier passage *right inside the prompt*, without learning
anything new (a simple form of in-context learning). Knock out the 5 strongest induction heads and the repeat rises
to more than 7 bits; knock out 5 random heads and it barely changes. Knocking out 2 heads isn't enough: the other
induction heads make up for them.

A detail worth noticing: 7.2, 5.1, 6.9 and 7.10 are both strong induction heads and members of the group that piles
onto the first token. In the Alice passage there is nothing repeated to copy, so they “park” on `<|endoftext|>`. The
same head does different jobs depending on its input.

### The distribution over all 50,257 tokens

At each generation step, GPT-2 gives 50,257 logits. The demo's file stores the 200 largest logits exactly, and the
remaining 50,057 logits are grouped into bins 0.01 wide (each token takes the logit at the middle of its bin). From
that we can rebuild the whole distribution at any temperature. The cell below rewrites the demo's computation, then
compares it with the numbers that `inside.py` computes exactly on all 50,257 logits.

In [ ]:
def classes(pr):
    """The logits in decreasing order and the number of tokens with each logit: the top 200 tokens (one class each), then the tail's bins."""
    tail = pr["tail"]
    z_tail = tail["lo"] + (np.array(tail["bins"][::-1]) + 0.5) * tail["bw"]
    z = np.concatenate([[zi for _, zi in pr["head"]], z_tail])
    return z, np.concatenate([np.ones(len(pr["head"])), tail["counts"][::-1]])


def dist_stats(pr, T):
    z, cnt = classes(pr)
    w = np.exp((z - z[0]) / T)
    p = w / (cnt * w).sum()  # the probability of ONE token in each class
    cum = np.cumsum(cnt * p)
    i = int(np.searchsorted(cum, 0.9))  # the class where the cumulative probability passes 0.9
    nucleus = int(cnt[:i].sum() + math.ceil((0.9 - (cum[i - 1] if i else 0)) / p[i] - 1e-12))
    entropy = -(cnt * p * np.log2(p)).sum()
    return {"top1": p[0], "top50": p[:50].sum(), "entropy": entropy, "nucleus90": nucleus}


for pr in samp["prompts"][:1] + samp["prompts"][-1:]:
    n_bins = len(pr["tail"]["bins"])
    print(f"“{pr['text']}”: {len(pr['head'])} exact logits + {num(pr['tail']['n'], 0)} logits in {num(n_bins, 0)} bins")
    for ex in pr["exact"]:
        st = dist_stats(pr, ex["T"])
        print(
            f"  T = {num(ex['T'], 1)}: top token {pct(st['top1'], 1):>6} · top 50 tokens {pct(st['top50'], 1):>6} ·"
            f" entropy {num(st['entropy'], 2):>5} bits · top-p 0.9 keeps {num(st['nucleus90'], 0):>6} tokens"
            f"   (exact: {pct(ex['top1'], 1)} · {pct(ex['top50'], 1)} · {num(ex['entropy'], 2)} · {num(ex['nucleus90'], 0)})"
        )
        assert abs(st["top50"] / ex["top50"] - 1) < 2e-3 and abs(st["entropy"] - ex["entropy"]) < 0.01

The rebuilt distribution matches the exact numbers to the last printed digit (the number of tokens top-p keeps at
high $T$ is off by 1, because the tokens in one bin share one logit). The two prompts show two things:
- **Temperature doesn't change the order**, only the sharpness: at $T = 0.5$, the top 50 tokens gather almost all the
  probability; at $T = 2$, those 50 tokens get only 16.1% (the Vietnamese prompt: 12.7%), and top-p 0.9 has to keep
  thousands of tokens. A large $T$ doesn't make the model more “creative”; it only gives the long tail of nearly
  meaningless tokens more chances to be sampled.
- **Top-p adapts by itself** to how certain the model is. At $T = 1$, the English prompt needs 294 tokens to gather
  0.9 of the probability, the Vietnamese prompt 225 tokens, even though the Vietnamese prompt's top token is only a
  byte fragment. Top-k, by contrast, always keeps exactly $k$ tokens.

The figure below plots the top 10 tokens after the Eiffel Tower prompt at three temperatures (solid, hatched and
dotted bars). The order of the tokens is identical at all three temperatures; only the lengths of the bars change.

In [ ]:
pr = next(q for q in samp["prompts"] if q["name"] == "eiffel")
z, cnt = classes(pr)
names = [TOK[i] for i, _ in pr["head"][:10]]
fig, ax = plt.subplots(figsize=(8, 4))
for j, (T, c, hatch) in enumerate(((0.5, BLUE, ""), (1.0, ORANGE, "///"), (2.0, GREEN, ".."))):
    w = np.exp((z - z[0]) / T)
    p10 = (w / (cnt * w).sum())[:10]
    ax.barh(
        np.arange(10) + (j - 1) * 0.27,
        p10,
        height=0.27,
        color=c,
        hatch=hatch,
        edgecolor="white",
        label=f"T = {num(T, 1)}",
    )
ax.set_yticks(range(10), [f"“{n_}”" for n_ in names])
ax.invert_yaxis()
ax.set(xlabel="probability")
ax.set_title(f"“{pr['text']} …”: top 10 tokens at three temperatures", fontsize=11)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
ax.legend(frameon=False)
plt.tight_layout()
fmt_axes(fig)
plt.show()

Finally, the passages GPT-2 actually writes as continuations (40 tokens, seed 0), with different sampling methods.
The number after “repeated” is the share of 4-token groups already seen earlier in the same passage: a simple measure
of repetition.

In [ ]:
for name in ("ex0", "eiffel"):
    pr = next(q for q in samp["prompts"] if q["name"] == name)
    print(f"“{pr['text']}”")
    for c in pr["cont"]:
        if c["seed"] == 0 and c["setting"] in ("greedy", "T0.7", "T1k40", "T2"):
            label = {"greedy": "greedy", "T0.7": "T = 0.7", "T1k40": "T = 1, top-k 40", "T2": "T = 2"}[c["setting"]]
            text_ = gpt_text(c["ids"]).replace("\n", " ⏎ ")
            print(f"  {label:<16} repeated {pct(c['rep4'], 0):>4} │ {text_[:110]}")

Greedy decoding falls into a loop: “The visualization is based on the following principles”, then the same again,
and for the Eiffel Tower prompt it repeats the prompt word for word; 30–56% of the 4-token groups are old ones.
$T = 2$ doesn't repeat, but it produces words glued together at random. In between, $T = 0.7$ or top-k 40 gives
fluent text (though not true: “Cologne, Germany”). The GPT-2 paper used exactly top-k 40 for its sample passages.
This is also why a *language model* is not yet an *assistant*: it continues a text plausibly; it doesn't answer
questions (Lesson 26).

## 13 · Try changing things

The cells below rerun the functions above with different parameters. You can edit the values and run them again.

**Knocking out parts of the small GPT.** Like the test on GPT-2: set the output of one sublayer to 0, or erase the
position table $P$, then measure the validation loss again (without retraining).

In [ ]:
def without(p, keys):
    return {k_: (np.zeros_like(a) if k_ in keys else a) for k_, a in p.items()}


print(f"every part: {num(gpt_bits, 3)} bits/character")
for l in range(CFG["L"]):
    print(f"without block {l}'s attention: {num(val_bits(without(params, {f'{l}.Wo', f'{l}.bo'}), CFG), 3)}", end=" · ")
    print(f"without block {l}'s MLP: {num(val_bits(without(params, {f'{l}.W2', f'{l}.c2'}), CFG), 3)}")
no_pos = val_bits(without(params, {"wpe"}), CFG)
print(f"position table P erased: {num(no_pos, 3)} (bigram: {num(bigram_bits, 3)})")
assert no_pos > gpt_bits + 0.5

d_ = CFG["d"]
no_bk = {k_: a.copy() for k_, a in params.items()}
for l in range(CFG["L"]):
    no_bk[f"{l}.bqkv"][d_ : 2 * d_] = 0  # set b_K = 0
bits_no_bk = val_bits(no_bk, CFG)
print(f"b_K after training: largest {num(np.abs(params['0.bqkv'][d_ : 2 * d_]).max(), 3)} (at the start: 0)")
print(f"b_K set to 0: {num(bits_no_bk, 4)} bits/character, against {num(gpt_bits, 4)}")
assert abs(bits_no_bk - gpt_bits) < 1e-4

As in GPT-2, the first block matters most: knocking out block 0's attention or MLP raises the loss by about 1 bit or
more, while knocking out a sublayer of block 1 adds only about 0.1 bit. Erasing $P$ makes the model even worse than
the bigram: it has learned to rely on position (for example, to know how many syllables into the line it is), and
now it has lost that information.

As for $b_K$: it starts at 0, its gradient is only float32 rounding noise, and yet after training it is no longer 0.
Adam divides the gradient by the square root of the mean square *of that same gradient*, so even very small noise
can turn into a sizable step. $b_K$ drifts away, but setting it back to 0 doesn't change the loss at all (it differs
only at the rounding level): as section 6 showed, it cannot affect the output.

**Temperature and loss.** Divide the logits by $T$ and measure the validation loss at several values of $T$. Which
$T$ gives the smallest loss?

In [ ]:
seq_v = np.concatenate([train_ids[-1:], val_ids])
Xv = seq_v[:8000].reshape(125, 64)
Yv = seq_v[1:8001].reshape(125, 64)
zv = forward(params, Xv, CFG)[0].astype(np.float64)
temps = [0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0]
bits_T = [cross_entropy(zv / T, Yv)[0] / math.log(2) for T in temps]
print(" · ".join(f"T = {num(T, 2)}: {num(b, 3)}" for T, b in zip(temps, bits_T)))
print(f"smallest at T = {num(temps[int(np.argmin(bits_T))], 2)}")

Smallest at $T$ near 1: the model was trained so that its own distribution ($T = 1$) fits the data. Lowering $T$
when generating text trades diversity for certainty; it doesn't make the model predict better.

**Other prompts and sampling methods.** Change `start`, `T` and `k`:

In [ ]:
start = "Đầu lòng hai ả tố nga,\n"  # try a line of your own (using only characters that occur in The Tale of Kiều)
for T, k in ((0.3, 0), (0.8, 10)):
    out_ = decode(generate(params, [stoi[c] for c in start], 150, CFG, T, k, seed=1))
    print(f"T = {num(T, 1)}, top-k {k or 'off'}:\n{start}{out_}\n")

**Retraining.** The cell below calls section 7's `train` again with a different configuration. By default it runs
only 65 steps to keep the notebook fast. Set `steps=650` and add the values 0.02, 0.03, 0.06 to `etas` to redo the
choice of $\eta$ yourself; or try `"L": 1`, `"heads": 1`, `"d": 64`. Each 650-step run takes about as long as
section 7. A different seed changes the initial weights and the order of the windows, so the results shift a little;
to compare two configurations, run a few seeds (`seeds=(0, 1, 2)`) and compare the spreads.

In [ ]:
etas, steps, seeds = (0.04,), 65, (0,)
for eta in etas:
    for seed in seeds:
        _, log_try = train({**CFG, "L": 2, "heads": 4}, steps=steps, eta=eta, seed=seed)
        print(f"η = {num(eta, 2)}, seed {seed}, {steps} steps: validation {num(log_try['val'][-1], 3)} bits/character")

After 65 steps, the model is only at about 3.5 bits, worse than the bigram; it takes the full 650 steps of section 7
for it to get down to 2.56 bits.

## 14 · Limits and next steps

**The same skeleton, with the details changed.** Open models such as Llama keep GPT-2's block but change four
details: RMSNorm instead of LayerNorm (Lesson 10), a gated SwiGLU MLP (three matrices instead of two), RoPE, which
rotates $q$ and $k$ according to position, instead of the table $P$ added to the input, and GQA, which lets several
query heads share one key/value pair. The cell below counts Llama 3 8B from its published configuration
(Grattafiori et al., 2024) the same way, and compares its KV cache with GPT-2's.

In [ ]:
def count_params(L, d, heads, kv_heads, hd, ff, vocab, ctx, swiglu, tied):
    q, kv = heads * hd, kv_heads * hd
    bias = not swiglu  # GPT-2 has biases and LayerNorm (g, b); Llama has no biases, and RMSNorm has only g
    attn = L * (d * (q + 2 * kv) + q * d + (q + 2 * kv + d if bias else 0))
    mlp = L * (3 * d * ff if swiglu else 2 * d * ff + ff + d)
    norm = (2 * L + 1) * (d if swiglu else 2 * d)
    pos = 0 if swiglu else ctx * d  # Llama: RoPE, no position table
    return vocab * d + pos + attn + mlp + norm + (0 if tied else vocab * d)


gpt2 = {"L": 12, "d": 768, "heads": 12, "kv_heads": 12, "hd": 64, "ff": 3072, "vocab": 50257, "ctx": 1024}
llama3 = {"L": 32, "d": 4096, "heads": 32, "kv_heads": 8, "hd": 128, "ff": 14336, "vocab": 128256, "ctx": 8192}
n_gpt2, n_llama = count_params(**gpt2, swiglu=False, tied=True), count_params(**llama3, swiglu=True, tied=False)
for name, c, n_ in (("GPT-2 small", gpt2, n_gpt2), ("Llama 3 8B", llama3, n_llama)):
    kv = 2 * c["L"] * c["kv_heads"] * c["hd"] * 2  # key + value, fp16
    mha = 2 * c["L"] * c["heads"] * c["hd"] * 2  # if every query head had its own K, V pair
    print(
        f"{name:<12} {num(n_, 0):>13} parameters · {c['heads']} query heads, {c['kv_heads']} KV heads · KV {num(kv / 1024, 0)} KiB"
        f"/token (without GQA: {num(mha / 1024, 0)}) · {num(c['ctx'], 0)} tokens: {num(kv * c['ctx'] / 2**30, 2)} GiB"
        f" (without GQA: {num(mha * c['ctx'] / 2**30, 2)})"
    )
assert n_gpt2 == 124439808 and n_llama == 8030261248  # matches the real GPT-2 weights and Hugging Face's Llama count

Llama 3 8B is about 65 times the size of GPT-2 small. Thanks to GQA (32 query heads, only 8 KV heads), each token
takes 128 KiB of cache instead of 512 KiB, and a sequence of 8,192 tokens takes 1 GiB instead of 4 GiB.

**What this notebook doesn't do yet:**
- The small GPT reads only 64 characters and has trained for 650 steps. Quality improves with model size, amount of
  data and compute, following laws that can be measured:
  [Lesson 23 · Scaling laws](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/quy-luat-mo-rong/) measures them with
  this very kind of character-level GPT on the Vietnamese Wikipedia.
- The cost of self-attention grows with the square of the sequence length, and the KV cache grows with the length:
  [Lesson 25](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn-tro-lai/) replaces it with a fixed-size state.
- A language model only continues text; turning it into an assistant is the job of
  [Lesson 26](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/).

Remove the mask and swap the text tokens for image patches, and we get the same Transformer block reading images:
the Vision Transformer, the next lesson.

## Summary
- GPT's input: $h_0[t] = E[x_t] + P[t]$. Self-attention: $\operatorname{softmax}(QK^\top/\sqrt{d_h} + M)\,V$, where
  the mask $M$ assigns $-\infty$ to the later positions; several heads run in parallel, are concatenated, then
  multiplied by $W_O$.
- The pre-LN block: $h \leftarrow h + \operatorname{MSA}(\operatorname{LN}(h))$, then $h \leftarrow h + \operatorname{MLP}(\operatorname{LN}(h))$.
  The handwritten backward pass matches finite differences; the gradient of $b_K$ is always 0.
- A small GPT with 72,516 parameters, 650 steps of AdamW: *The Tale of Kiều* from 7.04 down to 2.56 bits/character
  (bigram 3.14).
- With the mask, changing a later token doesn't change the earlier predictions at all, in both the small GPT and
  GPT-2 (Δ = 0). That is what lets one pass train every position.
- The KV cache gives exactly the result of the full pass. GPT-2: 36 KiB/token; generating 1,018 tokens is 352 times
  cheaper than recomputing.
- GPT-2 small has 124,439,808 parameters; the MLP takes up 2/3 of each block. Induction heads need two blocks: the
  2-block model can copy the repeated segment, the 1-block model cannot. In GPT-2, knocking out 5 induction heads
  pushes the repeat from 0.57 up to 7.71 bits.
- Temperature doesn't change the order of the tokens, only the sharpness; the validation loss is smallest at $T = 1$.
  Greedy decoding tends to repeat itself.

## Review questions
1. Why is the small GPT's initial loss exactly $\log_2 132$ bits, even though the embeddings and the attention
   matrices are random?
2. Edit the 20th token of the text. Why don't the predictions at the first 19 positions change? Why is the mask still
   needed in training, when the model already has the whole text?
3. Why does the key bias $b_K$ always have a gradient of 0?
4. The one-block model has 2 attention heads. Why can't it copy the repeated segment, while the two-block model can?
5. How many bytes does one GPT-2 token take in the KV cache (fp16)? Does the cache change the generated text?
6. Does raising the temperature $T$ change the top token? Why is the validation loss smallest at $T = 1$?

<details><summary>Hints and answers</summary>

1. The output layer's $W_h$ and $b_h$ start at 0, so every logit is 0 and softmax splits evenly over the 132
   characters.
2. With the mask, position $t$ reads only positions $j \le t$, so the 20th token doesn't enter the computation of
   any earlier position. In training, each position has to predict the next token; without the mask, it looks
   straight at the answer (the next token is right there in the input), learns to “copy the answer”, and is useless
   when generating text, when the future doesn't exist yet.
3. $b_K$ adds the same vector to every key, so every score in a row rises by the same amount $q_t \cdot b_K$;
   softmax doesn't change when all its inputs rise by the same amount.
4. To copy, the position holding [B] must already carry the information “before me is [A]”. That information exists
   only after one attention layer (the previous-token head), so a second attention layer is needed to find it. One
   block only knows which symbols have appeared.
5. $2 \times 12 \times 768 \times 2 = 36{,}864$ bytes (36 KiB). No: it gives exactly the logits of the full pass
   (differing only at the rounding level), because the keys and values of earlier positions don't depend on the new
   token.
6. No: dividing every logit by the same positive number doesn't change their order. The model was trained so that
   its own distribution at $T = 1$ fits the data; any other $T$ makes it overconfident or too hesitant.
</details>

## Further reading
- A. Vaswani et al. (2017), [Attention Is All You Need](https://arxiv.org/abs/1706.03762).
- A. Radford et al. (2019),
  [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
  (GPT-2), and [OpenAI's README](https://github.com/openai/gpt-2) with the correction 117M → 124M.
- A. Karpathy, [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY) and
  [nanoGPT](https://github.com/karpathy/nanoGPT): the same character-level GPT, written in PyTorch.
- J. Alammar, [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/).
- N. Elhage et al. (2021), [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html)
  and C. Olsson et al. (2022), [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html).
- nostalgebraist (2020), [interpreting GPT: the logit lens](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens);
  N. Belrose et al. (2023), [tuned lens](https://arxiv.org/abs/2303.08112).
- G. Xiao et al. (2023), [Efficient Streaming Language Models with Attention Sinks](https://arxiv.org/abs/2309.17453).
- A. Holtzman et al. (2019), [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751) (top-p).
- A. Haviv et al. (2022), [Transformer Language Models without Positional Encodings Still Learn Positional
  Information](https://arxiv.org/abs/2203.16634).
- J. Kaplan et al. (2020), [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) (≈ 2N FLOPs per token).
- Polo Club, [Transformer Explainer](https://poloclub.github.io/transformer-explainer): GPT-2 running in the browser, the reel's source.

**Next lesson:** [Lesson 18 · Vision Transformer (ViT)](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vision-transformer/):
the same Transformer block, without a mask, reading an image as a sequence of patches.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")